# Parallel-safety test for `QASARetriever`

`QASARetriever` carries activation state in a Cypher `MAP` variable rather than a
graph property, so concurrent retrievals operate on independent state and cannot
corrupt each other. This notebook verifies the two claims behind that design:

1. **Concurrent correctness** — running the retriever from multiple threads
   produces results identical to serial execution. We compare an order-independent
   fingerprint (seeds, entities, paths) against a serial baseline for every worker
   count.
2. **Speedup** — wall-clock time decreases as the worker count increases.

A retriever that mutated a shared graph property (e.g. `_resource`) would have
concurrent queries corrupt each other's state; this notebook demonstrates that
`QASARetriever` does not.

In [ ]:
import os
import sys
import time
import pickle
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

from dotenv import load_dotenv
from neo4j import GraphDatabase

sys.path.insert(0, str(Path("..").resolve()))
load_dotenv("../.env")

In [ ]:
from qasa_rag.embedder import Embedder
from qasa_rag.retrieval import QASARetriever

NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password123")

DATASET_NAME = "musique"
N_QUESTIONS = 30
MAX_WORKERS = 8

driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USER, NEO4J_PASSWORD),
    max_connection_pool_size=MAX_WORKERS + 4,
)
embedder = Embedder(cache_path=Path("cache/embeddings_cache.pkl"))

QUERY_ENTITY_CACHE_PATH = Path(f"cache/query_entities-{DATASET_NAME}.pkl")
if QUERY_ENTITY_CACHE_PATH.exists():
    with open(QUERY_ENTITY_CACHE_PATH, "rb") as f:
        query_entity_cache = pickle.load(f)
    print(f"[QueryEntityCache] Loaded {len(query_entity_cache)} cached query entities")
else:
    query_entity_cache = {}
    print("[QueryEntityCache] No cache found, starting empty")

In [ ]:
with open(f"ground_truth-{DATASET_NAME}.pkl", "rb") as f:
    ground_truth = pickle.load(f)

questions = [gt["question"] for gt in ground_truth[:N_QUESTIONS]]
print(f"Testing on {len(questions)} questions from {DATASET_NAME}")

retriever_kwargs = dict(
    driver=driver,
    embedder=embedder,
    max_steps=3,
    decay=0.7,
    resource_threshold=0.01,
    seed_vector_fallback_k=5,
    top_k_entities=30,
    top_k_paths=30,
    query_entity_cache=query_entity_cache,
)

stateless_retriever = QASARetriever(**retriever_kwargs)

### Step 1: warm up caches

Run each question once serially to populate the query-entity LLM cache and the embedder cache, so the timing comparison below isolates Neo4j retrieval cost rather than LLM extraction cost.

In [ ]:
missing = [q for q in questions if q not in query_entity_cache]
if missing:
    print(f"Warming up {len(missing)} questions (calling LLM)...")
    for q in missing:
        stateless_retriever._extract_query_entities(q)
    QUERY_ENTITY_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    with open(QUERY_ENTITY_CACHE_PATH, "wb") as f:
        pickle.dump(query_entity_cache, f)
    print(f"Cached {len(missing)} new entries")
else:
    print("All questions already cached — LLM will not be called below")

### Step 2: serial baseline

Run every question serially to record a reference fingerprint and timing. The
concurrent runs in Step 3 must reproduce these fingerprints exactly. The
fingerprint is order-independent (it sorts seeds, entities and paths), so it is
insensitive to Cypher's tie-breaking.

In [ ]:
def fingerprint(result):
    """Order-independent signature of a RetrievalResult for equality testing."""
    return (
        tuple(sorted(result.seed_entities)),
        tuple(sorted(e["name"] for e in result.entities)),
        tuple(sorted(tuple(p["names"]) for p in result.paths)),
    )


t0 = time.perf_counter()
stateless_results = [stateless_retriever.retrieve(q) for q in questions]
stateless_elapsed = time.perf_counter() - t0
stateless_fingerprints = [fingerprint(r) for r in stateless_results]

print(f"Serial baseline: {stateless_elapsed:6.2f}s  ({stateless_elapsed / len(questions):.3f}s/q)")
print(f"Collected {len(stateless_fingerprints)} reference fingerprints")

### Step 3: concurrent execution of the stateless retriever

Compare against the serial stateless baseline measured above. Each worker count must produce fingerprints identical to that baseline.

In [ ]:
def run_parallel(retriever, questions, max_workers):
    results = [None] * len(questions)
    t0 = time.perf_counter()
    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        future_to_idx = {pool.submit(retriever.retrieve, q): i for i, q in enumerate(questions)}
        for fut in as_completed(future_to_idx):
            idx = future_to_idx[fut]
            results[idx] = fut.result()
    elapsed = time.perf_counter() - t0
    return results, elapsed


rows = []
for workers in (1, 2, 4, 8):
    parallel_results, parallel_elapsed = run_parallel(stateless_retriever, questions, workers)
    parallel_fingerprints = [fingerprint(r) for r in parallel_results]
    matches = sum(s == p for s, p in zip(stateless_fingerprints, parallel_fingerprints))
    speedup = stateless_elapsed / parallel_elapsed
    rows.append({
        "workers": workers,
        "elapsed_s": round(parallel_elapsed, 2),
        "speedup_vs_serial": round(speedup, 2),
        "matches_serial": f"{matches}/{len(questions)}",
    })
    print(f"  workers={workers:2d}  {parallel_elapsed:6.2f}s  speedup={speedup:.2f}x  matches={matches}/{len(questions)}")

### Summary

In [ ]:
import pandas as pd

df = pd.DataFrame(rows)
df.insert(0, "mode", df["workers"].apply(lambda w: "serial" if w == 1 else f"parallel x{w}"))
print(df.to_string(index=False))

parallel_ok = all(row["matches_serial"].split("/")[0] == str(len(questions)) for row in rows)

print(f"\n[Parallel correctness]  all worker counts reproduce the serial fingerprints  {'OK' if parallel_ok else 'FAIL'}")

if parallel_ok:
    print("\nQASARetriever is verified parallel-safe: concurrent execution yields results "
          "identical to serial, with a wall-clock speedup as workers increase.")
else:
    print("\nWARNING: parallel results diverged from the serial baseline. Investigate.")

In [ ]:
driver.close()